# Northwind Data Visualization - Interactive Dashboard

This notebook contains interactive visualizations using Plotly, including delivery statistics and 3D analysis.

In [10]:
# Install required packages if not already installed
!pip install plotly nbformat pandas --quiet

In [11]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Ensure figures dir exists
os.makedirs("../figures", exist_ok=True)

ModuleNotFoundError: No module named 'plotly'

In [ ]:
# Load the warehouse data
data_path = "../data/warehouse/merged_northwind.csv"
if not os.path.exists(data_path):
    print(f"File not found: {data_path}. Please run the ETL scripts first.")
else:
    df = pd.read_csv(data_path)
    df['FullDate'] = pd.to_datetime(df['FullDate'])
    print("Data loaded successfully.")
    print(f"Total records: {len(df)}")
    print(df.head())

## 1. Delivery Status Overview

In [ ]:
if 'df' in locals():
    delivery_counts = df['DeliveredFlag'].value_counts()
    
    fig = go.Figure(data=[go.Pie(
        labels=['Delivered', 'Not Delivered'],
        values=[delivery_counts.get(1, 0), delivery_counts.get(0, 0)],
        hole=0.3,
        marker=dict(colors=['#2ecc71', '#e74c3c']),
        textinfo='label+percent+value',
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    )])
    
    fig.update_layout(
        title='Order Delivery Status',
        font=dict(size=14),
        height=500
    )
    
    try:
        fig.write_html("../figures/delivery_stats_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")

## 2. Orders by Country (Interactive)

In [ ]:
if 'df' in locals():
    country_orders = df['Country_x'].value_counts().reset_index()
    country_orders.columns = ['Country', 'OrderCount']
    
    fig = px.bar(
        country_orders,
        x='Country',
        y='OrderCount',
        title='Total Orders by Country',
        color='OrderCount',
        color_continuous_scale='Viridis',
        hover_data={'OrderCount': ':,'}
    )
    
    fig.update_layout(
        xaxis_title='Country',
        yaxis_title='Number of Orders',
        height=600
    )
    
    try:
        fig.write_html("../figures/orders_by_country_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")

## 3. Orders by Employee (Interactive)

In [ ]:
if 'df' in locals():
    df['EmployeeName'] = df['FirstName'] + ' ' + df['LastName']
    employee_orders = df['EmployeeName'].value_counts().reset_index()
    employee_orders.columns = ['EmployeeName', 'OrderCount']
    
    fig = px.bar(
        employee_orders,
        y='EmployeeName',
        x='OrderCount',
        orientation='h',
        title='Orders by Employee',
        color='OrderCount',
        color_continuous_scale='Plasma'
    )
    
    fig.update_layout(
        xaxis_title='Number of Orders',
        yaxis_title='Employee',
        height=600
    )
    
    try:
        fig.write_html("../figures/orders_by_employee_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")

## 4. Monthly Orders Trend (Interactive)

In [ ]:
if 'df' in locals():
    df['YearMonth'] = df['FullDate'].dt.to_period('M').astype(str)
    monthly_orders = df.groupby('YearMonth').size().reset_index(name='OrderCount')
    
    fig = px.line(
        monthly_orders,
        x='YearMonth',
        y='OrderCount',
        title='Monthly Orders Trend',
        markers=True
    )
    
    fig.update_traces(
        line=dict(color='#3498db', width=3),
        marker=dict(size=8)
    )
    
    fig.update_layout(
        xaxis_title='Month',
        yaxis_title='Number of Orders',
        height=600,
        hovermode='x unified'
    )
    
    try:
        fig.write_html("../figures/monthly_trend_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")

## 5. 3D Interactive Visualization: Orders by Month and Country

In [ ]:
if 'df' in locals():
    df['MonthNum'] = df['FullDate'].dt.month
    agg = df.groupby(['MonthNum', 'Country_x']).size().reset_index(name='OrderCount')
    
    fig = px.scatter_3d(
        agg,
        x='MonthNum',
        y='Country_x',
        z='OrderCount',
        color='OrderCount',
        size='OrderCount',
        title='3D View: Orders by Month and Country',
        color_continuous_scale='Turbo',
        hover_data={'MonthNum': True, 'Country_x': True, 'OrderCount': True}
    )
    
    fig.update_layout(
        scene=dict(
            xaxis_title='Month',
            yaxis_title='Country',
            zaxis_title='Order Count'
        ),
        height=700
    )
    
    try:
        fig.write_html("../figures/3d_orders_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")

## 6. Delivery Status by Country

In [ ]:
if 'df' in locals():
    delivery_by_country = df.groupby(['Country_x', 'DeliveredFlag']).size().reset_index(name='Count')
    delivery_by_country['Status'] = delivery_by_country['DeliveredFlag'].map({1: 'Delivered', 0: 'Not Delivered'})
    
    fig = px.bar(
        delivery_by_country,
        x='Country_x',
        y='Count',
        color='Status',
        title='Delivery Status by Country',
        barmode='stack',
        color_discrete_map={'Delivered': '#2ecc71', 'Not Delivered': '#e74c3c'}
    )
    
    fig.update_layout(
        xaxis_title='Country',
        yaxis_title='Number of Orders',
        height=600
    )
    
    try:
        fig.write_html("../figures/delivery_by_country_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")

## 7. Comprehensive Dashboard

In [ ]:
if 'df' in locals():
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Delivery Status', 'Orders by Country', 'Monthly Trend', 'Top Employees'),
        specs=[[{'type': 'pie'}, {'type': 'bar'}],
               [{'type': 'scatter'}, {'type': 'bar'}]]
    )
    
    # Delivery Status Pie
    delivery_counts = df['DeliveredFlag'].value_counts()
    fig.add_trace(
        go.Pie(labels=['Delivered', 'Not Delivered'],
               values=[delivery_counts.get(1, 0), delivery_counts.get(0, 0)],
               marker=dict(colors=['#2ecc71', '#e74c3c'])),
        row=1, col=1
    )
    
    # Orders by Country
    country_orders = df['Country_x'].value_counts().head(5).reset_index()
    country_orders.columns = ['Country', 'Count']
    fig.add_trace(
        go.Bar(x=country_orders['Country'], y=country_orders['Count'],
               marker=dict(color='#3498db')),
        row=1, col=2
    )
    
    # Monthly Trend
    df['YearMonth'] = df['FullDate'].dt.to_period('M').astype(str)
    monthly = df.groupby('YearMonth').size().reset_index(name='Count')
    fig.add_trace(
        go.Scatter(x=monthly['YearMonth'], y=monthly['Count'],
                   mode='lines+markers', line=dict(color='#9b59b6')),
        row=2, col=1
    )
    
    # Top Employees
    df['EmployeeName'] = df['FirstName'] + ' ' + df['LastName']
    employee_orders = df['EmployeeName'].value_counts().head(5).reset_index()
    employee_orders.columns = ['Employee', 'Count']
    fig.add_trace(
        go.Bar(y=employee_orders['Employee'], x=employee_orders['Count'],
               orientation='h', marker=dict(color='#e67e22')),
        row=2, col=2
    )
    
    fig.update_layout(
        title_text='Northwind Orders Dashboard',
        showlegend=False,
        height=900
    )
    
    try:
        fig.write_html("../figures/dashboard_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")